# VLQ and Sirius connection test
One Bell circuit (100 shots) on FakeVLQ, then on real VLQ, then on IQM Sirius. Set `VLQ_PROJECT` (and optionally `VLQ_RESOURCE`, `VLQ_TOKEN`) in `.env`; for Sirius put the IQM Resonance token in `.IQM_token` or `IQM_TOKEN`.

In [ ]:
from dotenv import load_dotenv
from qiskit import QuantumCircuit

from utils import vlq

load_dotenv()
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.measure_all()

In [ ]:
fake = vlq.fake_vlq()
tr, layout = vlq.transpile([bell], fake, seed=1)
print(layout, vlq.run(fake, tr, 100, seed=1))

In [ ]:
# Real hardware from here on. Without VLQ_TOKEN this starts an interactive LEXIS login.
from qheappe_client import QProvider
import os

provider = QProvider(lexis_project=os.environ["VLQ_PROJECT"], token=vlq.get_token())  # prints a login URL: open it
provider.available_backends(os.environ.get("VLQ_RESOURCE") or None)

In [ ]:
backend = vlq.vlq_backend()
print(backend.name, backend.num_qubits, backend.architecture.calibration_set_id)
tr, layout = vlq.transpile([bell], backend)
print(layout, vlq.run(backend, tr, 100))  # expect mostly 00 and 11

In [ ]:
# IQM Sirius (IQM Resonance)
sirius = vlq.sirius_backend()
print(sirius.name, sirius.num_qubits, sirius.architecture.calibration_set_id)
tr, layout = vlq.transpile([bell], sirius)
print(layout, vlq.run(sirius, tr, 100))  # expect mostly 00 and 11